**Ankle Jump-Landing Model (OpenSim)**

3-segment foot-ankle model for simulating jump landing with prescribed ground reaction impulse.

**File Overview**

| Section | Purpose |
|---------|---------|
| **Bodies** | 4 segments: tibia, talus, calcaneus, forefoot |
| **Joints** | TT (plantarflex/dorsiflex), ST (inversion/eversion), MT (flex/ext) |
| **Ligament limits** | `CoordinateLimitForce` restricts joint ROM |
| **Talocrural spring** | PF/DF muscle-equivalent (800 Nm/rad) |
| **Subtalar spring** | Inversion/eversion (Asmussen 2022; default 0.1 Nm/kg/deg) |
| **Prescribed force** | 3D ground-reaction impulse on forefoot (peak 2×BW, ramps to residual) |
| **run_once** | Single simulation; writes row to `day2_runs.csv` |
| **run_batch** | Sweeps `T_PEAK_S`, `RESID_FRAC`, impact angles |
| **SURFACE_CONDITIONS** | Asmussen (2022): flat, lateral_ridge, medial_ridge (NR, LR, MR) |
| **run_asmussen_sweep** | Sweeps subtalar stiffness 0.1→1.6 Nm/kg/deg (± surface angles) |
| **run_mentor_protocol** | Full mentor protocol: 3 surfaces × 5 stiffnesses; CSV column surface |

**Mentor feedback & references**

**From mentor's email:**  
*"Here is the Asmussen (2022) paper with Fig 3 showing the ankle muscles dynamically acting like a spring about the subtalar joint during running on a flat surface and when landing on a flat surface, a lateral and a medial ridge under the foot. The average spring stiffness looks to be **0.1 Nm/kg body weight/deg** (see the PPT file attached). So you could assume the subtalar joint spring stiffness is a **minimum 0.1 Nm/kg/deg when running**, but then could **increase it in the model of a jump landing from 0.1 to 0.2, 0.4, 0.8, and 1.6 Nm/kg/deg** to see how intentionally cocontracting the medial and lateral muscles about the joint prior to ground impact can affect the joint response when landing on something unusual under the foot."*

**Reference:**  
Asmussen MJ, Lichtwark GA, Maharaj JN. The Subtalar Joint Maintains "Spring-Like" Function While Running in Footwear That Perturbs Foot Pronation. *J Appl Biomech*. 2022;38(4):221–231.  
- **Fig 3:** joint angle vs moment → slope = rotational (quasi-)stiffness; stiffness did not differ across conditions (~0.1 Nm/kg/deg).  
- **Conditions:** no ridge (NR, flat), lateral ridge (LR, promotes pronation), medial ridge (MR, reduces pronation).  
- Tibialis posterior activation varied with condition; spring-like energy absorb/return in early/late stance.

**Imports**

In [1]:
import opensim as osim
import math
import os, csv
import numpy as np

**Basic Numeric Setup**

**Asmussen (2022) surface conditions**

Landing surfaces from the paper (Fig 1): **no ridge (NR)**, **lateral ridge (LR)**, **medial ridge (MR)**.  
We approximate them via the 3-D impact direction: flat = vertical; LR/MR = added lateral/medial component (tune angles as needed).

In [2]:
SURFACE_NR = (0, 0, 0)
SURFACE_LR = (12, 0, 0)
SURFACE_MR = (-12, 0, 0)
SURFACE_CONDITIONS = {"flat": SURFACE_NR, "lateral_ridge": SURFACE_LR, "medial_ridge": SURFACE_MR}

In [3]:
BW_N = 700.0
T_PEAK_S = 0.060
RESID_FRAC = 0.50
SIM_T = 0.20

ANGLE_X = 0.0
ANGLE_Y = 0.0
ANGLE_Z = 0.0

ST_STIFFNESS_NM_KG_DEG = 0.1

SURFACE_LABEL = ""

RESULTS_PATH = os.path.join(os.getcwd(), "day2_runs.csv")

**Model Function**

In [4]:
def build_model():
    model = osim.Model()
    model.setName("Day2_Ankle_3D")
    ground = model.getGround()

    tibia = osim.Body("tibia", 3.0, osim.Vec3(0), osim.Inertia(0.02,0.02,0.02))
    talus = osim.Body("talus", 0.2, osim.Vec3(0), osim.Inertia(0.001,0.001,0.001))
    calc  = osim.Body("calcaneus", 0.3, osim.Vec3(0), osim.Inertia(0.002,0.002,0.002))
    fore  = osim.Body("forefoot", 0.2, osim.Vec3(0), osim.Inertia(0.001,0.001,0.001))
    for b in (tibia, talus, calc, fore):
        model.addBody(b)

    model.addJoint(osim.WeldJoint("tibia_ground",
        ground, osim.Vec3(0), osim.Vec3(0), tibia, osim.Vec3(0), osim.Vec3(0)))

    TT = osim.PinJoint("TT", tibia, osim.Vec3(0), osim.Vec3(0),
                       talus, osim.Vec3(0), osim.Vec3(0))
    TT.upd_coordinates(0).setName("tt_pfdf")
    model.addJoint(TT)

    ST = osim.PinJoint("ST", talus, osim.Vec3(0), osim.Vec3(0),
                       calc, osim.Vec3(0), osim.Vec3(0))
    ST.upd_coordinates(0).setName("st_inv_ev")
    model.addJoint(ST)

    MT = osim.PinJoint("MT", calc, osim.Vec3(0), osim.Vec3(0),
                       fore, osim.Vec3(0), osim.Vec3(0))
    MT.upd_coordinates(0).setName("mt_flex_ext")
    model.addJoint(MT)

    model.addForce(osim.CoordinateLimitForce(
        "tt_pfdf", 30*math.pi/180, 1e3, -20*math.pi/180, 1e3, 5, 0.05))
    model.addForce(osim.CoordinateLimitForce(
        "st_inv_ev", 30*math.pi/180, 1e3, -20*math.pi/180, 1e3, 5, 0.05))
    model.addForce(osim.CoordinateLimitForce(
        "mt_flex_ext", 15*math.pi/180, 500, -15*math.pi/180, 500, 3, 0.05))

    pf_spring = osim.SpringGeneralizedForce()
    pf_spring.set_coordinate("tt_pfdf")
    pf_spring.set_stiffness(800.0)
    pf_spring.set_rest_length(0.0)
    pf_spring.set_viscosity(10.0)
    model.addForce(pf_spring)

    mass_kg = BW_N / 9.81
    k_st_nm_rad = ST_STIFFNESS_NM_KG_DEG * mass_kg * (180.0 / math.pi)
    st_spring = osim.SpringGeneralizedForce()
    st_spring.set_coordinate("st_inv_ev")
    st_spring.set_stiffness(k_st_nm_rad)
    st_spring.set_rest_length(0.0)
    st_spring.set_viscosity(5.0)
    model.addForce(st_spring)

    force = osim.PrescribedForce("forefoot_force", fore)
    force.set_pointIsGlobal(True)
    force.set_forceIsGlobal(True)

    times = [0.0, T_PEAK_S, SIM_T]
    Fy = [0.0, 2.0*BW_N, RESID_FRAC*BW_N]
    fn_y = osim.PiecewiseLinearFunction()
    for t, v in zip(times, Fy):
        fn_y.addPoint(t, v)

    dir_x, dir_y, dir_z = math.sin(ANGLE_X), math.cos(ANGLE_Y), math.sin(ANGLE_Z)
    norm = math.sqrt(dir_x**2 + dir_y**2 + dir_z**2)
    dir_x, dir_y, dir_z = dir_x/norm, dir_y/norm, dir_z/norm

    fn_x = osim.PiecewiseLinearFunction()
    fn_z = osim.PiecewiseLinearFunction()
    for t, v in zip(times, Fy):
        fn_x.addPoint(t, v*dir_x)
        fn_z.addPoint(t, v*dir_z)

    funcs = osim.FunctionSet()
    funcs.cloneAndAppend(fn_x)
    funcs.cloneAndAppend(fn_y)
    funcs.cloneAndAppend(fn_z)
    force.set_forceFunctions(funcs)
    model.addForce(force)

    model.finalizeConnections()
    return model

**Run Single Simulation**

In [5]:
def run_once():
    model = build_model()
    state = model.initSystem()

    cs = model.updCoordinateSet()
    cs.get("tt_pfdf").setValue(state, -5*math.pi/180)
    cs.get("st_inv_ev").setValue(state, -2*math.pi/180)
    cs.get("mt_flex_ext").setValue(state, 0)

    manager = osim.Manager(model)
    state.setTime(0.0)
    manager.setIntegratorAccuracy(1e-3)
    manager.initialize(state)
    final_state = manager.integrate(SIM_T)

    tt = cs.get("tt_pfdf").getValue(final_state)
    st = cs.get("st_inv_ev").getValue(final_state)
    mt = cs.get("mt_flex_ext").getValue(final_state)

    failed = (
        (tt > 30*math.pi/180) or (tt < -20*math.pi/180) or
        (abs(st) > 30*math.pi/180) or (abs(mt) > 15*math.pi/180)
    )

    row = {
        "t_peak_ms": int(T_PEAK_S*1000),
        "residual_frac": RESID_FRAC,
        "angle_x_deg": int(math.degrees(ANGLE_X)),
        "angle_y_deg": int(math.degrees(ANGLE_Y)),
        "angle_z_deg": int(math.degrees(ANGLE_Z)),
        "st_stiffness_nm_kg_deg": ST_STIFFNESS_NM_KG_DEG,
        "surface": SURFACE_LABEL,
        "tt_deg_final": tt*180/math.pi,
        "st_deg_final": st*180/math.pi,
        "mt_deg_final": mt*180/math.pi,
        "failed": int(failed)
    }

    print(row)
    os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)
    write_header = not os.path.exists(RESULTS_PATH)
    with open(RESULTS_PATH, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=row.keys())
        if write_header: w.writeheader()
        w.writerow(row)

**Batch Runs (Parameter Sweep)**

In [6]:
def run_batch():
    t_peaks = [0.04, 0.06]
    residuals = [0.25, 0.50]
    angles_deg = [0, 25]

    total = len(t_peaks)*len(residuals)*len(angles_deg)**3
    count = 0
    for tp in t_peaks:
        for rf in residuals:
            for ax in angles_deg:
                for ay in angles_deg:
                    for az in angles_deg:
                        count += 1
                        global T_PEAK_S, RESID_FRAC, ANGLE_X, ANGLE_Y, ANGLE_Z
                        T_PEAK_S, RESID_FRAC = tp, rf
                        ANGLE_X, ANGLE_Y, ANGLE_Z = map(math.radians, [ax, ay, az])
                        print(f"[{count}/{total}] tp={tp:.3f}s rf={rf:.2f} angles=({ax},{ay},{az})")
                        run_once()

**Execute**

---
**Asmussen (2022) Implementation: Subtalar Joint Spring Stiffness**

Per professor feedback: *"assume subtalar joint spring stiffness is a minimum 0.1 Nm/kg/deg when running, but increase it (0.1, 0.2, 0.4, 0.8, 1.6 Nm/kg/deg) to see how cocontracting medial/lateral muscles prior to impact affects joint response when landing on unusual surfaces."*

**Conversion:** `k_Nm/rad = k_Nm/(kg·deg) × mass_kg × (180/π)`

In [7]:
STIFFNESS_NM_KG_DEG = [0.1, 0.2, 0.4, 0.8, 1.6]

def stiffness_nm_per_kg_per_deg_to_nm_per_rad(k_deg, mass_kg):
    return k_deg * mass_kg * (180.0 / math.pi)

mass_kg = BW_N / 9.81
for k in STIFFNESS_NM_KG_DEG:
    print(f"{k} Nm/(kg·deg) → {stiffness_nm_per_kg_per_deg_to_nm_per_rad(k, mass_kg):.1f} Nm/rad")

def run_asmussen_sweep(surface_angles=None):
    if surface_angles is None:
        surface_angles = [(0, 0, 0)]
    global ST_STIFFNESS_NM_KG_DEG, ANGLE_X, ANGLE_Y, ANGLE_Z, SURFACE_LABEL
    SURFACE_LABEL = ""
    for ax, ay, az in surface_angles:
        ANGLE_X, ANGLE_Y, ANGLE_Z = map(math.radians, [ax, ay, az])
        for k in STIFFNESS_NM_KG_DEG:
            ST_STIFFNESS_NM_KG_DEG = k
            print(f"st_stiff={k} Nm/(kg·deg), angles=({ax},{ay},{az})")
            run_once()

def run_mentor_protocol():
    global ANGLE_X, ANGLE_Y, ANGLE_Z, ST_STIFFNESS_NM_KG_DEG, SURFACE_LABEL
    for name, (ax, ay, az) in SURFACE_CONDITIONS.items():
        SURFACE_LABEL = name
        ANGLE_X, ANGLE_Y, ANGLE_Z = map(math.radians, [ax, ay, az])
        for k in STIFFNESS_NM_KG_DEG:
            ST_STIFFNESS_NM_KG_DEG = k
            print(f"surface={name}, st_stiff={k} Nm/(kg·deg)")
            run_once()
    SURFACE_LABEL = ""

0.1 Nm/(kg·deg) → 408.8 Nm/rad
0.2 Nm/(kg·deg) → 817.7 Nm/rad
0.4 Nm/(kg·deg) → 1635.4 Nm/rad
0.8 Nm/(kg·deg) → 3270.7 Nm/rad
1.6 Nm/(kg·deg) → 6541.4 Nm/rad


In [8]:
run_once()

[warning] ForceSet 'forceset' has subcomponents with duplicate name 'springgeneralizedforce'. The duplicate is being renamed to 'springgeneralizedforce_0'.
{'t_peak_ms': 60, 'residual_frac': 0.5, 'angle_x_deg': 0, 'angle_y_deg': 0, 'angle_z_deg': 0, 'st_stiffness_nm_kg_deg': 0.1, 'surface': '', 'tt_deg_final': -4.200478755892307e-06, 'st_deg_final': 4.816184048145801e-06, 'mt_deg_final': 0.2619125096564505, 'failed': 0}
